# [9665] Locality-Sensitive Hashing 2
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Netflix_titles.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/17/25 12:35:38


### Import libraries

In [ ]:
%%time

! pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 2.8 MB/s eta 0:00:00
CPU times: user 82.4 ms, sys: 17.2 ms, total: 99.6 ms
Wall time: 9.52 s


In [ ]:
import numpy as np
import pandas as pd
import time
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from datasketch import MinHash
from datasketch import MinHashLSHForest

In [ ]:
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Netflix_titles.csv')

### Examine data

In [ ]:
df.shape

(7787, 12)

In [ ]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df['description'].head(5)

,description
0,"In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to join the 3% saved from squalor."
1,"After a devastating earthquake hits Mexico City, trapped survivors from all walks of life wait to be rescued while trying desperately to stay alive."
2,"When an army recruit is found dead, his fellow soldiers are forced to confront a terrifying secret that's haunting their jungle island training camp."
3,"In a postapocalyptic world, rag-doll robots hide in fear from dangerous machines out to exterminate them, until a brave newcomer joins the group."
4,A brilliant group of students become card-counting experts with the intent of swindling millions out of Las Vegas casinos by playing blackjack.


### Create function to preprocess text

In [ ]:
# Function to clean_text
def clean_text(text):
    lem = WordNetLemmatizer()
    stop = set(stopwords.words('english'))
    punct = string.punctuation
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', punct)).lower()
    tokens = re.split(r'\W+', text)
    tokens = [lem.lemmatize(word) for word in tokens if word not in stop]
    return ' '.join(tokens)

In [ ]:
%%time

# Preprocess (clean) new column 'combined_text'
df['description_clean'] = df['description'].apply(clean_text)
df.head()

CPU times: user 7.49 s, sys: 474 ms, total: 7.96 s
Wall time: 14.2 s


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,description_clean
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, Rodolfo Valente, Vaneza Oliveira, Rafael Lozano, Viviane Porto, Mel Fronckowiak, Sergio Mamberti, Zezé Motta, Celso Frateschi",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi & Fantasy","In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to join the 3% saved from squalor.",future elite inhabit island paradise far crowded slum get one chance join 3 saved squalor
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, Azalia Ortiz, Octavio Michel, Carmen Beato",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies","After a devastating earthquake hits Mexico City, trapped survivors from all walks of life wait to be rescued while trying desperately to stay alive.",devastating earthquake hit mexico city trapped survivor walk life wait rescued trying desperately stay alive
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence Koh, Tommy Kuan, Josh Lai, Mark Lee, Susan Leong, Benjamin Lim",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow soldiers are forced to confront a terrifying secret that's haunting their jungle island training camp.",army recruit found dead fellow soldier forced confront terrifying secret thats haunting jungle island training camp
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly, Christopher Plummer, Crispin Glover, Martin Landau, Fred Tatasciore, Alan Oppenheimer, Tom Kane",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi & Fantasy","In a postapocalyptic world, rag-doll robots hide in fear from dangerous machines out to exterminate them, until a brave newcomer joins the group.",postapocalyptic world ragdoll robot hide fear dangerous machine exterminate brave newcomer join group
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aaron Yoo, Liza Lapira, Jacob Pitts, Laurence Fishburne, Jack McGee, Josh Gad, Sam Golzari, Helen Carey, Jack Gilpin",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-counting experts with the intent of swindling millions out of Las Vegas casinos by playing blackjack.,brilliant group student become cardcounting expert intent swindling million la vega casino playing blackjack


In [ ]:
df[['description','description_clean']].head()

,description,description_clean
0,"In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to join the 3% saved from squalor.",future elite inhabit island paradise far crowded slum get one chance join 3 saved squalor
1,"After a devastating earthquake hits Mexico City, trapped survivors from all walks of life wait to be rescued while trying desperately to stay alive.",devastating earthquake hit mexico city trapped survivor walk life wait rescued trying desperately stay alive
2,"When an army recruit is found dead, his fellow soldiers are forced to confront a terrifying secret that's haunting their jungle island training camp.",army recruit found dead fellow soldier forced confront terrifying secret thats haunting jungle island training camp
3,"In a postapocalyptic world, rag-doll robots hide in fear from dangerous machines out to exterminate them, until a brave newcomer joins the group.",postapocalyptic world ragdoll robot hide fear dangerous machine exterminate brave newcomer join group
4,A brilliant group of students become card-counting experts with the intent of swindling millions out of Las Vegas casinos by playing blackjack.,brilliant group student become cardcounting expert intent swindling million la vega casino playing blackjack


In [ ]:
# Split data into training and validation sets.  Reserve 0.01% of data to test with later.
df1, df2 = train_test_split(df, test_size=.005, random_state=42)

In [ ]:
df1.shape

(7748, 13)

In [ ]:
df2.shape

(39, 13)

## Part 1
### Shingle is determined by word boundary

### Create function to generate MinHash Forest
* Initialize number of permutations in MinHash
* MinHash the string on all shingles in each document
* Store the MinHash of the string
* Generate a forest of all MinHashed strings
* Index the forest to make it searchable

In [ ]:
def generate_forest(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        m = MinHash(num_perm=permutations)
        for token in doc:                      # Process shingles on word boundary
            m.update(token.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

### Create function to query MinHash Forest
* Preprocess input text into shingles
* Use the same number of permutations for the MinHash as was used to build the forest
* Create a MinHash on the input text using all shingles
* Query the forest with MinHash and return the number of requested recommendations
* Provide the titles of each conference paper recommended

In [ ]:
def predict(text, df, permutations, num_results, forest):
    start_time = time.time()

#    tokens = clean_text(text)
    m = MinHash(num_perm=permutations)
    for token in text:
        m.update(token.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest

In [ ]:
# Set number of Permutations
permutations = 128

In [ ]:
forest = generate_forest(df1['description_clean'], permutations)

It took 29.90922522544861 seconds to build forest.


### Make recommendation

In [ ]:
idx = 7
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.0029363632202148438 seconds to query forest.

Top 5 recommendations for [An American Tail: The Mystery of the Night Monster]:
5317      Rust Valley Restorers
7730               You vs. Wild
4840    Pettersson and Findus 2
2114                     Faraar
2995         Inequality for All
Name: title, dtype: object


## Part 2
### Shingle size is fixed

In [ ]:
def create_shingles(text, shingle_size=6):
    return [text[i:i+shingle_size] for i in range(len(text)-shingle_size+1)]

In [ ]:
def generate_forest_2(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        shingles = create_shingles(doc)
        m = MinHash(num_perm=permutations)
        for shingle in shingles:
            m.update(shingle.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

In [ ]:
def predict_2(text, df, permutations, num_results, forest):
    start_time = time.time()

    tokens = clean_text(text)
    shingles = create_shingles(tokens)
    m = MinHash(num_perm=permutations)
    for shingle in shingles:
        m.update(shingle.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest

In [ ]:
forest = generate_forest_2(df1['description_clean'], permutations)

It took 22.222254991531372 seconds to build forest.


### Make recommendations

In [ ]:
idx = 7
num_recommendations = 5
#title = 'Using a neural net to instantiate a deformable model'
#title = 'Statistical Analysis of Semi-Supervised Learning: The Limit of Infinite Unlabelled Data'
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict_2(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.006518125534057617 seconds to query forest.

Top 5 recommendations for [An American Tail: The Mystery of the Night Monster]:
227     A Shaun the Sheep Movie: Farmageddon
4195                   Monster High: Haunted
4030                             Memory Love
4125                           Mirror Mirror
2319                        Furthest Witness
Name: title, dtype: object
